# 03 — Golden Dataset EDA: Hallucination-Free Patient Summaries

**Source**: Hegselmann et al., *"A Data-Centric Approach To Generate Faithful
and High Quality Patient Summaries with Large Language Models"* (CHIL 2024)

**PhysioNet**: `medical-expert-annotations-of-unsupported-facts-in-doctor-
written-and-llm-generated-patient-summaries-1.0.1`

## Purpose

Explore the **cleaned & improved** (golden) dataset — 100 doctor-written
patient summaries from MIMIC-IV where two medical experts:

1. Annotated all unsupported facts (hallucinations) using an 11-label protocol
2. Manually removed or replaced every hallucination
3. Further improved the summaries for quality

**Goal**: Understand what makes these summaries "hallucination-free" to select
good few-shot examples for the thesis experiments.

## Dataset Lineage

```
MIMIC-IV-Note (discharge notes)
  └─ MIMIC-IV-Note-Ext-DI (100,175 context-summary pairs)
      └─ MIMIC-IV-Note-Ext-DI-BHC (same, Brief Hospital Course as context)
          └─ *_4000_600_chars (26,178 pairs, context ≤4000, summary ≥600)
              └─ 100 random pairs → annotated by 2 medical experts
                  ├─ hallucinations_mimic_di.jsonl (with hallucination labels)
                  ├─ derived/original.json       (raw doctor-written)
                  ├─ derived/cleaned.json         (hallucinations removed)
                  └─ derived/cleaned_improved.json (cleaned + quality improved)
                                                   ← THIS IS THE GOLDEN DATASET
```

In [1]:
import json
from pathlib import Path
from collections import Counter
from textwrap import fill

# ── Paths ──
DATA_ROOT = Path("../../data/raw/medical-expert-annotations-of-unsupported-facts-"
                 "in-doctor-written-and-llm-generated-patient-summaries-1.0.1")

HALLUC_DIR = DATA_ROOT / "hallucination_datasets"
DERIVED_DIR = DATA_ROOT / "derived_datasets"

# ── Load all 3 stages ──
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

golden    = load_jsonl(DERIVED_DIR / "hallucinations_mimic_di_cleaned_improved.json")
cleaned   = load_jsonl(DERIVED_DIR / "hallucinations_mimic_di_cleaned.json")
original  = load_jsonl(DERIVED_DIR / "hallucinations_mimic_di_original.json")
annotated = load_jsonl(HALLUC_DIR / "hallucinations_mimic_di.jsonl")

print(f"Original (doctor-written):   {len(original)} records")
print(f"Cleaned (halluc removed):    {len(cleaned)} records")
print(f"Golden (cleaned+improved):   {len(golden)} records")
print(f"Annotated (with labels):     {len(annotated)} records")

Original (doctor-written):   100 records
Cleaned (halluc removed):    100 records
Golden (cleaned+improved):   100 records
Annotated (with labels):     100 records


## 1. Data Structure

Each record has two fields:
- **`text`** — the **Brief Hospital Course** (BHC) section from the discharge
  note. This is the clinical context written by doctors for medical staff.
- **`summary`** — the **Discharge Instructions** (DI) section, rewritten as a
  patient-facing summary. This is what patients receive upon leaving.

The annotated version adds a **`labels`** field with span-level hallucination
annotations.

In [2]:
# Show the schema
print("=== Golden dataset record keys ===")
print(list(golden[0].keys()))
print()
print("=== Annotated dataset record keys ===")
print(list(annotated[0].keys()))

=== Golden dataset record keys ===
['text', 'summary']

=== Annotated dataset record keys ===
['text', 'summary', 'labels']


## 2. Full Example: Context → Golden Summary

Let's look at a complete pair to understand the relationship between the BHC
(clinical notes for doctors) and the DI (patient-facing summary).

In [3]:
def show_pair(records, idx, title=""):
    """Display a context-summary pair with clear formatting."""
    r = records[idx]
    text = r["text"]
    summary = r["summary"]
    print(f"{'='*70}")
    print(f"  {title} — Record #{idx}")
    print(f"  Context length: {len(text):,} chars | Summary length: {len(summary):,} chars")
    print(f"{'='*70}")
    print()
    print("── CONTEXT (Brief Hospital Course) ──")
    print()
    print(fill(text, width=80))
    print()
    print("── SUMMARY (Discharge Instructions — Golden) ──")
    print()
    print(fill(summary, width=80))
    print()

show_pair(golden, 0, "Sample A: Short case (double vision)")

  Sample A: Short case (double vision) — Record #0
  Context length: 1,245 chars | Summary length: 736 chars

── CONTEXT (Brief Hospital Course) ──

Brief Hospital Course: Mr. ___ was admitted on ___ for double vision. On
arrival, he had a CT scan that revealed no signs of an acute intracranial
process. An MRI/MRA was completed for further evaluation of ischemia. However,
there were no findings on MRI c/w acute infarct. We consulted ophthamology and
they suspect that his vision loss was secondary to an ischemic event despite no
radiologic findings. We increased his aspirin from 81mg to 325mg daily during
his hospitalization. We prescribed artificial tears four times a day per
ophthamology recommendations. He was referred to neuro-ophthamology as an
outpatient. He was also scheduled for an ECHO as an outpatient within the next
___ hours. During his hospitalization, he was started on an insulin sliding
scale. He did have one episode of hypoglycemia to 38. However, he was given
dextrose. 

In [4]:
show_pair(golden, 50, "Sample B: Complex case (unresponsiveness + fall)")

  Sample B: Complex case (unresponsiveness + fall) — Record #50
  Context length: 2,641 chars | Summary length: 680 chars

── CONTEXT (Brief Hospital Course) ──

Brief Hospital Course: Mr. ___ is a ___ yo RH man with history of CAD s/p CABG,
HTN, HLD, DMII, history of multiple strokes/TIAs in the past and gait disorder
and recent hospitalization for a small stroke with discharge to rehab facility
who now presents with an episode of unresponsiveness followed by lethargy after
a fall, initially concerning for hemorrhage but CT without evidence of bleeding.
Some initial concern for seizure given EEG on previous admission showing
slowing/epileptiform discharges from L temporal lobe #NEURO: On initial exam,
mental status was notable for non-fluent speech, and orientation to person but
not time or place. Patient appeared withdrawn on admission. Physical exam was
notable for mild right NLFF (chronic) as well as weakness in Right upper
extremity as well as lower extremities bilaterally R>L wor

In [5]:
show_pair(golden, 99, "Sample C: Multi-problem case (nausea, aortic dissection)")

  Sample C: Multi-problem case (nausea, aortic dissection) — Record #99
  Context length: 3,564 chars | Summary length: 690 chars

── CONTEXT (Brief Hospital Course) ──

Brief Hospital Course: ___ yo M with a history of AAA and Type A aortic
dissection presenting for nausea/vomiting with chest and epigastric pain. ACUTE
# Emesis. Patient describes nonbloody, occasionally bilious vomiting in the
setting of esophageal "burning" and marked leukocytosis. MRI of his aorta showed
stable AAA and stable dissection without evidence of any other intra-abdominal
processes. Labs showed stable hct at 51, troponin < .01, and lipase WNL. The
most likely etiology was thought to be a viral gastroenteritis given headaches,
chills, and diarrhea, but given history of improvement with warm baths, this
could also represent cannibis emesis syndrome. He was managed with zofran and
given maintenance IV fluids on the floor, with improvement in his symptoms and
tolerating POs at the time of his discharge. # ___.

## 3. How Hallucinations Were Removed: Original → Cleaned → Improved

The paper defines 11 hallucination types. Let's see exactly what was changed
at each stage for a single record. This is key to understanding what makes
the golden dataset "faithful."

In [6]:
def show_evolution(idx):
    """Show how a summary evolves from original → cleaned → improved."""
    orig_text = original[idx]["summary"]
    clean_text = cleaned[idx]["summary"]
    gold_text = golden[idx]["summary"]

    # Get hallucination annotations for this record
    anno = annotated[idx]
    labels = anno.get("labels", [])

    print(f"{'='*70}")
    print(f"  Record #{idx} — Evolution through 3 stages")
    print(f"  Context: {len(anno['text']):,} chars | Hallucinations found: {len(labels)}")
    print(f"{'='*70}")
    print()

    if labels:
        print("── HALLUCINATIONS FOUND ──")
        for i, lab in enumerate(labels, 1):
            print(f"  {i}. [{lab['label']}] \"{lab['text']}\"")
            print(f"     Position: chars {lab['start']}–{lab['end']} ({lab['length']} chars)")
        print()

    print(f"── ORIGINAL (doctor-written, {len(orig_text)} chars) ──")
    print(fill(orig_text, width=80))
    print()

    print(f"── CLEANED (hallucinations removed, {len(clean_text)} chars) ──")
    print(fill(clean_text, width=80))
    print()

    print(f"── GOLDEN (cleaned + improved, {len(gold_text)} chars) ──")
    print(fill(gold_text, width=80))
    print()

    # Highlight differences
    if orig_text != clean_text:
        diff_chars = len(orig_text) - len(clean_text)
        print(f"  📝 Cleaning removed {diff_chars} chars ({diff_chars/len(orig_text)*100:.1f}%)")
    if clean_text != gold_text:
        diff_chars = len(clean_text) - len(gold_text)
        print(f"  ✨ Improvement changed {abs(diff_chars)} chars "
              f"({'removed' if diff_chars > 0 else 'added'})")
    if orig_text == clean_text == gold_text:
        print("  ✅ No changes — original was already hallucination-free!")

# Record 0 has 3 hallucinations — good example of cleaning
show_evolution(0)

  Record #0 — Evolution through 3 stages
  Context: 1,245 chars | Hallucinations found: 5

── HALLUCINATIONS FOUND ──
  1. [location_unsupported] "to one of the nerves supplying the muscles to the eye"
     Position: chars 100–153 (53 chars)
  2. [location_unsupported] "to both eyes"
     Position: chars 700–712 (12 chars)
  3. [medication_unsupported] "metoprolol"
     Position: chars 761–771 (10 chars)
  4. [medication_unsupported] "simvastatin"
     Position: chars 776–787 (11 chars)
  5. [word_unsupported] "as an inpatient"
     Position: chars 820–835 (15 chars)

── ORIGINAL (doctor-written, 1010 chars) ──
You were admitted after you developed double vision. We suspect that you may
have an ischemic event to one of the nerves supplying the muscles to the eye.
During your hospitalization, you had a CT scan and an MRI completed that did not
reveal any signs of an acute stroke. The ophthomalogist evaluated you during
your hospital admission and they suspect that your symptoms may have

In [7]:
# Find a record with many hallucinations to show dramatic cleaning
most_halluc_idx = max(range(len(annotated)),
                      key=lambda i: len(annotated[i].get("labels", [])))
print(f"Record with most hallucinations: #{most_halluc_idx} "
      f"({len(annotated[most_halluc_idx]['labels'])} annotations)")
print()
show_evolution(most_halluc_idx)

Record with most hallucinations: #7 (10 annotations)

  Record #7 — Evolution through 3 stages
  Context: 1,367 chars | Hallucinations found: 10

── HALLUCINATIONS FOUND ──
  1. [time_unsupported] "acute onset"
     Position: chars 39–50 (11 chars)
  2. [location_unsupported] "back"
     Position: chars 51–55 (4 chars)
  3. [word_unsupported] "Spine"
     Position: chars 159–164 (5 chars)
  4. [condition_unsupported] "difficulty walking"
     Position: chars 336–354 (18 chars)
  5. [word_unsupported] "for"
     Position: chars 386–389 (3 chars)
  6. [word_unsupported] "procedure was delayed"
     Position: chars 417–438 (21 chars)
  7. [word_unsupported] "lab value indicating a higher risk of bleeding"
     Position: chars 448–494 (46 chars)
  8. [medication_unsupported] "attributed to antibiotics"
     Position: chars 505–530 (25 chars)
  9. [medication_unsupported] "treated with vitamin K"
     Position: chars 535–557 (22 chars)
  10. [time_unsupported] "on ___"
     Position: chars 

In [8]:
# Show a record with 0 hallucinations (originally clean)
zero_halluc_indices = [i for i, r in enumerate(annotated)
                       if len(r.get("labels", [])) == 0]
print(f"Records with 0 hallucinations: {len(zero_halluc_indices)}")
print(f"Indices: {zero_halluc_indices}")
print()
show_evolution(zero_halluc_indices[0])

Records with 0 hallucinations: 7
Indices: [2, 37, 53, 57, 70, 71, 95]

  Record #2 — Evolution through 3 stages
  Context: 3,592 chars | Hallucinations found: 0

── ORIGINAL (doctor-written, 738 chars) ──
You were admitted to the hospital with an intestinal obstruction. You were taken
to the operating room an underwent lysis of abdominal adhesive scar tissue
bands. You had a nasogastric tube placed to help with bowel decompression. You
started having return of bowel function and this tube was removed, but you later
became nauseated with vomiting and had the nasogastric tube replaced. The tube
has been removed again and you are now tolerating a regular diet. While in the
hospital, you were diagnosed with epididymitis, an infection of the duct behind
the testis. You were started on a course antibiotics which you will complete at
home. You have already had this medication filled at the bedside and require two
more days of treatment.

── CLEANED (hallucinations removed, 738 chars) ──
You w

## 4. The 11 Hallucination Label Types

The paper's annotation protocol defines 11 categories of unsupported facts.
Understanding these is critical for designing few-shot prompts that teach
models what to avoid.

In [9]:
# Collect all labels from annotated dataset
all_labels = []
for r in annotated:
    for lab in r.get("labels", []):
        all_labels.append(lab)

label_counts = Counter(lab["label"] for lab in all_labels)

print(f"Total hallucination annotations: {len(all_labels)} across {len(annotated)} summaries")
print(f"Avg hallucinations per summary: {len(all_labels)/len(annotated):.1f}")
print()
print("Label type distribution:")
print(f"{'Label':<30} {'Count':>5}  Examples")
print("-" * 70)

# Group examples by label for display
label_examples = {}
for lab in all_labels:
    label_examples.setdefault(lab["label"], []).append(lab["text"])

for label, count in label_counts.most_common():
    examples = label_examples[label][:2]
    examples_str = " | ".join(f'"{e[:40]}"' for e in examples)
    print(f"{label:<30} {count:>5}  {examples_str}")

Total hallucination annotations: 286 across 100 summaries
Avg hallucinations per summary: 2.9

Label type distribution:
Label                          Count  Examples
----------------------------------------------------------------------
word_unsupported                  76  "as an inpatient" | "a result from"
condition_unsupported             52  "cumulative injury to the brain from hype" | "lightheadedness"
time_unsupported                  35  "on ___" | "acute onset"
medication_unsupported            34  "metoprolol" | "simvastatin"
location_unsupported              29  "to one of the nerves supplying the muscl" | "to both eyes"
procedure_unsupported             19  "upper endoscopy" | "EKG"
name_unsupported                  18  "Oncologist" | "pancreas GI team"
contradicted_fact                 15  "anemia" | "not"
number_unsupported                 7  "20" | "132"
other_unsupported                  1  "Hi Mr. ___, You were admitted for nausea"


### Label Descriptions (from the annotation protocol)

| Label | Description | Clinical Risk |
|-------|-------------|---------------|
| `word_unsupported` | A single word that is not in the source | Medium |
| `condition_unsupported` | Medical condition mentioned without support | **High** |
| `medication_unsupported` | Medication mentioned without support | **High** |
| `time_unsupported` | Temporal reference not in the source | Medium |
| `location_unsupported` | Anatomical/spatial detail unsupported | Medium |
| `procedure_unsupported` | Medical procedure not mentioned in source | **High** |
| `name_unsupported` | Name/identifier not in source | Low |
| `contradicted_fact` | Directly contradicts the source text | **Critical** |
| `number_unsupported` | Numeric value not supported by source | **High** |
| `other_unsupported` | Catch-all for other unsupported facts | Variable |
| `reasoning_unsupported` | Clinical reasoning not in source | Medium |

## 5. Content Patterns in Golden Summaries

Let's understand the **structure and tone** of the golden summaries — this is
what we want the model to learn via few-shot examples.

In [10]:
print("=== How golden summaries typically begin ===")
print()
for i, r in enumerate(golden[:15]):
    first_sentence = r["summary"].split(".")[0] + "."
    print(f"  [{i:2d}] {first_sentence[:100]}")

=== How golden summaries typically begin ===

  [ 0] You were admitted after you developed double vision.
  [ 1] You were admitted to the hospital with spells that are consistent with seizures.
  [ 2] You were admitted to the hospital with an intestinal obstruction.
  [ 3] You were admitted to the hospital to characterize events concerning for seizure.
  [ 4] The patient was admitted after a fall.
  [ 5] You were admitted to the hospital for shortness of breath, abdominal pain, and elevated liver number
  [ 6] You were admitted to the epilepsy service for video-EEG monitoring to capture your spells of dizzine
  [ 7] You were admitted to the hospital with leg pain that we feel is likely due to a herniated disk in yo
  [ 8] You were admitted after feeling faint because of a heart rhythm called atrial fibrillation.
  [ 9] You were admitted for chest pain.
  [10] You were started on blood thinners and were found to have a bleed as a result of those blood thinner
  [11] You came to the hosp

In [11]:
print("=== How golden summaries typically end ===")
print()
for i, r in enumerate(golden[:15]):
    last_sentence = r["summary"].rstrip().rsplit(".", 2)[-2] + "." if "." in r["summary"] else r["summary"][-100:]
    print(f"  [{i:2d}] ...{last_sentence.strip()[:100]}")

=== How golden summaries typically end ===

  [ 0] ...We ordered an ECHO as an outpatient.
  [ 1] ...climbing ladders, taking baths alone, or operating heavy machinery - for the next several months in 
  [ 2] ...You have already had this medication filled at the bedside and require two more days of treatment.
  [ 3] ...In order to reduce the risk of side effects, this medication was started at a low dose and will need
  [ 4] ...Given his declining medical condition, a family meeting was had and he was made comfort measures onl
  [ 5] ...We did stop your doxepin which may have been causing some of your side effects.
  [ 6] ...In 1 week you should increase this dose to 300 mg at night.
  [ 7] ...You finally had the epidural injection.
  [ 8] ...Please have a follow up with your cardiologist since we have adjusted your flecainide.
  [ 9] ...We obtained an MRI of your pelvis which showed normal placement of the catheter.
  [10] ...You had drainage of the fluid in your lungs.
  [11] ...We a

### Key Observations for Prompt Design

The golden summaries follow consistent patterns:

1. **Patient-facing language**: "You were admitted for..." (2nd person)
2. **Clinical accuracy**: Only facts present in BHC, no inference
3. **Structure**: Admission reason → Findings → Treatment → Discharge plan
4. **De-identified tokens**: `___` preserved (names, dates, etc.)
5. **Professional but accessible**: Medical terms are used but explained

## 6. Relationship Between Context and Summary

Understanding how the BHC context maps to the DI summary helps us design
prompts that instruct models on the expected transformation.

In [12]:
# Show the text-to-summary relationship
print("=== Context → Summary Mapping Patterns ===\n")

for idx in [0, 10, 30, 50, 75]:
    r = golden[idx]
    text = r["text"]
    summary = r["summary"]

    # Check if summary starts with "You were"
    starts_with_you = summary.strip().startswith("You")
    # Check for de-id tokens
    has_deident = "___" in summary
    # Check for section headers in context
    has_sections = any(tag in text for tag in ["#", "ACUTE", "CHRONIC", "ASSESSMENT"])

    print(f"Record #{idx}:")
    print(f"  Context: {len(text):,} chars | Summary: {len(summary):,} chars "
          f"| Ratio: {len(summary)/len(text):.2f}")
    print(f"  Starts with 'You': {starts_with_you} | Has ___: {has_deident} "
          f"| Context has sections: {has_sections}")
    print(f"  Context begins: \"{text[:80]}...\"")
    print(f"  Summary begins: \"{summary[:80]}...\"")
    print()

=== Context → Summary Mapping Patterns ===

Record #0:
  Context: 1,245 chars | Summary: 736 chars | Ratio: 0.59
  Starts with 'You': True | Has ___: False | Context has sections: False
  Context begins: "Brief Hospital Course: Mr. ___ was admitted on ___ for double vision. On arrival..."
  Summary begins: "You were admitted after you developed double vision. We suspect that you may hav..."

Record #10:
  Context: 3,964 chars | Summary: 568 chars | Ratio: 0.14
  Starts with 'You': True | Has ___: False | Context has sections: True
  Context begins: "Brief Hospital Course: ___ woman with a history of mild diabetes and osteoarthri..."
  Summary begins: "You were started on blood thinners and were found to have a bleed as a result of..."

Record #30:
  Context: 2,749 chars | Summary: 1,187 chars | Ratio: 0.43
  Starts with 'You': True | Has ___: False | Context has sections: True
  Context begins: "Brief Hospital Course: Assessment and Plan: ___ yo with RUQ pain, transaminitis ..."
  Summ

## 7. Candidate Selection for Few-Shot Prompting

For few-shot prompting, we want examples that are:
- **Representative**: typical BHC → DI transformation
- **Clean**: no remaining artifacts or ambiguity
- **Reasonable length**: short enough to fit in prompt budget
- **Diverse**: cover different medical conditions/complexities

Let's identify candidates.

In [13]:
# Score each record for few-shot suitability
candidates = []
for i, r in enumerate(golden):
    text = r["text"]
    summary = r["summary"]

    # Get original hallucination count
    orig_halluc_count = len(annotated[i].get("labels", [])) if i < len(annotated) else 0

    record_info = {
        "idx": i,
        "text_len": len(text),
        "summary_len": len(summary),
        "ratio": len(summary) / len(text) if len(text) > 0 else 0,
        "starts_with_you": summary.strip().startswith("You"),
        "orig_halluc_count": orig_halluc_count,
        "context_first_line": text.split("\n")[0][:100],
        "summary_first_sentence": summary.split(".")[0][:100],
    }
    candidates.append(record_info)

# Sort by combined length (text + summary) — shorter = better for few-shot
candidates_sorted = sorted(candidates, key=lambda x: x["text_len"] + x["summary_len"])

print("=== Top 15 shortest pairs (best for few-shot prompt budget) ===\n")
print(f"{'Idx':>4} {'Text':>6} {'Summ':>6} {'Ratio':>6} {'Orig H':>7} First line of context")
print("-" * 90)
for c in candidates_sorted[:15]:
    print(f"{c['idx']:4d} {c['text_len']:6d} {c['summary_len']:6d} "
          f"{c['ratio']:6.2f} {c['orig_halluc_count']:7d}  "
          f"{c['context_first_line'][:50]}")

=== Top 15 shortest pairs (best for few-shot prompt budget) ===

 Idx   Text   Summ  Ratio  Orig H First line of context
------------------------------------------------------------------------------------------
   6    696    553   0.79       3  Brief Hospital Course: Ms. ___ was admitted to the
   8    763    562   0.74       9  Brief Hospital Course: ___ with h/o paroxysmal afi
  52    844    564   0.67       5  Brief Hospital Course: ___ with no significant med
  20    911    657   0.72       5  Brief Hospital Course: ___ year old female with hi
  56   1055    596   0.56       6  Brief Hospital Course: The patient is a ___ year o
  34   1085    635   0.59       2  Brief Hospital Course: Ms. ___ is a ___ year old f
  76   1257    487   0.39       4  Brief Hospital Course: Ms. ___ is a ___ year old w
  87   1115    640   0.57       1  Brief Hospital Course: Mrs. ___ is a ___ yo right 
   3    925    854   0.92       1  Brief Hospital Course: Ms. ___ is a ___ year old w
   7   1367   

In [14]:
# Show the top 5 shortest complete pairs
print("=== Top 5 Shortest Golden Pairs (Full Content) ===\n")
for rank, c in enumerate(candidates_sorted[:5], 1):
    idx = c["idx"]
    r = golden[idx]
    print(f"{'='*70}")
    print(f"  RANK #{rank} — Record #{idx}")
    print(f"  Context: {c['text_len']} chars | Summary: {c['summary_len']} chars")
    print(f"  Originally had {c['orig_halluc_count']} hallucination(s)")
    print(f"{'='*70}")
    print()
    print("── CONTEXT ──")
    print(fill(r["text"], width=80))
    print()
    print("── GOLDEN SUMMARY ──")
    print(fill(r["summary"], width=80))
    print()

=== Top 5 Shortest Golden Pairs (Full Content) ===

  RANK #1 — Record #6
  Context: 696 chars | Summary: 553 chars
  Originally had 3 hallucination(s)

── CONTEXT ──
Brief Hospital Course: Ms. ___ was admitted to the epilepsy service from ___.
She had screening bloodwork and urine analysis on admission and this was
unremarkable. She was monitored on video-EEE throughout the hospitalization and
her Keppra was slowly weaned off. She had several typical episodes of dizziness,
nausea and falling to the ground during her stay. None of these had an EEG
correlate. She was transitioned to zonegran with a dose of 200mg qHS. She had
orthostatic blood pressures and glucose monitored during the spells and these
were unremarkable. Her florineff was discontinued and her blood pressures
remained stable. The plan is to increase her zonegran in one week to 300mg qHS.

── GOLDEN SUMMARY ──
You were admitted to the epilepsy service for video-EEG monitoring to capture
your spells of dizziness and nausea.

## 8. Full Catalog: All 100 Golden Records at a Glance

Quick reference showing the medical topic and dimensions for each record.

In [15]:
print(f"{'Idx':>3}  {'Text':>5} {'Summ':>5} {'H':>2}  First line of BHC context")
print("-" * 95)
for i, r in enumerate(golden):
    first_line = r["text"].split("\n")[0].replace("Brief Hospital Course: ", "")[:65]
    h_count = len(annotated[i].get("labels", [])) if i < len(annotated) else "?"
    print(f"{i:3d}  {len(r['text']):5d} {len(r['summary']):5d} {h_count:>2}  {first_line}")

Idx   Text  Summ  H  First line of BHC context
-----------------------------------------------------------------------------------------------
  0   1245   736  5  Mr. ___ was admitted on ___ for double vision. On arrival, he had
  1   2561   708  2  Mr. ___ is a ___ old right-handed male with a history of CAD, HTN
  2   3592   741  0  Mr. ___ is a ___ year-old male with a history of congential diaph
  3    925   854  1  Ms. ___ is a ___ year old woman with a past medical history signi
  4   2545   638  1  Deceased on ___ at 6:55pm Mr ___ is a ___ yr old male with hx of 
  5   2818  1038  3  Mr. ___ is a ___ male with history of multiple myeloma on Daratum
  6    696   553  3  Ms. ___ was admitted to the epilepsy service from ___. She had sc
  7   1367   542 10  Ms. ___ is a ___ year old woman with a history of rheumatoid arth
  8    763   562  9  ___ with h/o paroxysmal afib who presented for dc cardioversion f
  9   1823   671  3  ___ w/ hx of NSTEMI p/w atypical and typical chest pa

## 9. Summary of Findings

### What is the Golden Dataset?
- **100 doctor-written Discharge Instructions** from MIMIC-IV
- Each paired with its **Brief Hospital Course** as source context
- All hallucinations **manually removed** by two medical experts
- Further **improved** for quality and readability

### Why it matters for the thesis
- These are **verified hallucination-free** summaries
- They demonstrate the **exact transformation** (BHC → DI) we want models to
  learn
- They can serve as **few-shot examples** in the prompt to teach models:
  1. Use patient-facing language ("You were admitted for...")
  2. Only include facts from the source BHC
  3. Follow the structure: admission → findings → treatment → discharge
  4. Preserve de-identified tokens (`___`)

### Few-shot candidate criteria
- **Short pairs** (< 2000 chars total) to fit within prompt budget
- **Had hallucinations originally** (shows the model what "corrected" looks
  like vs. the original)
- **Representative medical topics** (not all the same condition)

### Next steps
- Select 2–3 few-shot examples from the candidates
- Design a `BaseTechnique` subclass that injects these into the prompt
- Compare against baseline (zero-shot) across all models and ranges